# 02. Causal discovery and validation

この notebook は因果探索と、因果探索時・stage 接続時の validation を扱います。

扱うもの:

- discovery algorithm: `pc`, `ges`, `lingam`, `notears`
- `CausalDiscovery` の責務
- background knowledge と temporal tier
- discovery result / edge artifact
- cross-stage validation

重要: 因果探索は graph 仮説を出します。graph edge をそのまま treatment effect と読むべきではありません。

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

REPOSITORY_MARKER = "pyproject.toml"
cwd = Path.cwd().resolve()
for candidate in (cwd, *cwd.parents):
    if (candidate / REPOSITORY_MARKER).exists():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("repository root was not found")

ARTICLE_ROOT = PROJECT_ROOT
SRC_DIR = PROJECT_ROOT / "src"
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
for path in (SRC_DIR,):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

ARTICLE_ROOT

## Algorithm selection

| algorithm | 背景 | 実装上の注意 |
|---|---|---|
| `pc` | constraint-based。条件付き独立性から graph を絞る | `causal-learn`。独立性検定に依存 |
| `ges` | score-based。score を改善する graph を探索する | `causal-learn` |
| `lingam` | 非ガウス性などの仮定に基づく線形非巡回モデル | optional `lingam` package が必要 |
| `notears` | DAG constraint を連続最適化で扱う | optional `notears` 実装が必要 |

同じデータでも、algorithm と仮定が違えば graph は変わります。したがって、探索結果は「発見された真実」ではなく「仮定つきの候補」です。

In [ ]:
from causal_atelier.shared.constants import SUPPORTED_DISCOVERY_ALGORITHMS
from causal_atelier.infrastructure.config import load_yaml_mapping

analysis_path = PROJECT_ROOT / "configs" / "causal" / "discovery.yaml"
analysis_config = load_yaml_mapping(analysis_path)

pd.DataFrame([{
    "supported_algorithms": list(SUPPORTED_DISCOVERY_ALGORITHMS),
    "configured_algorithms": analysis_config["discovery"]["algorithms"],
    "pc_indep_test": analysis_config["discovery"]["pc"]["indep_test"],
}])

## CausalDiscovery を小さな合成データで動かす

ここでは実データを使わず、`x -> y` と `z -> y` があるような合成データを作ります。目的は causal-learn の PC を完全に理解することではなく、このコード体系では discovery result がどのように edge table へ正規化されるかを見ることです。

In [ ]:
from causal_atelier.causal.discovery.algorithms import CausalDiscovery
from causal_atelier.preprocessing.discovery.config import load_feature_config

rng = np.random.default_rng(7)
n = 300
x = rng.normal(size=n)
z = rng.normal(size=n)
y = 0.8 * x + 0.5 * z + rng.normal(scale=0.4, size=n)
frame = pd.DataFrame({"x": x, "z": z, "y": y})
frame = (frame - frame.mean(axis=0)) / frame.std(axis=0)

feature_config = load_feature_config(PROJECT_ROOT / "configs" / "preprocessing" / "discovery_features.yaml")
discovery = CausalDiscovery(
    alpha=0.05,
    use_background_knowledge=False,
    feature_config=feature_config,
    algorithms=("pc",),
    bootstrap_samples=0,
)
results = discovery.run_all(frame)
pd.DataFrame([
    {"algorithm": name, "status": result.status, "edges": len(result.edges), "message": result.message}
    for name, result in results.items()
])

In [ ]:
results["pc"].edges

## Discovery 時の validation

因果探索で validation できることは限定的です。

検証できること:

- algorithm 名が allowlist に入っているか。
- PC の independent test が許可値か。
- bootstrap や discretization parameter が範囲内か。
- feature config の source column、transform、background tier が定義されているか。
- inference stage に渡す manifest schema が最低限成立するか。

検証できないこと:

- causal sufficiency が成り立つか。
- すべての交絡が観測されているか。
- PC/GES/LiNGAM/NOTEARS の仮定が実データで正しいか。
- 出てきた edge が政策介入効果を表すか。

In [ ]:
from causal_atelier.interfaces.cli.pipeline import parse_args
from causal_atelier.application.planning import PipelinePlanner
from causal_atelier.application.validation import CrossStageValidator

args = parse_args([
    "--project-root", str(PROJECT_ROOT),
    "--discovery-algorithms", "pc",
    "--discovery-output-dir", str(PROJECT_ROOT / "artifacts" / "experiments" / "causal_discovery"),
    "--inference-output-dir", str(PROJECT_ROOT / "artifacts" / "experiments" / "causal_inference"),
])
plan = PipelinePlanner(PROJECT_ROOT).build_plan(args, strategy_name="validate_only")
validation = CrossStageValidator().validate(plan)
pd.DataFrame(validation.to_dicts()) if validation.issues else pd.DataFrame([{"validation": "ok"}])

## Background knowledge

background knowledge は、feature config の temporal tier を使って時間的に不自然な edge を抑制します。

Why so?: 観測データだけから向きを完全に決めるのは難しいため、キャンペーン前・キャンペーン中・キャンペーン後という時間順序を制約として使います。

反対仮説: background knowledge が間違っていれば、正しい edge を禁止してしまう可能性があります。したがって、これは統計的 tuning ではなく causal design の一部です。